# Task 4.1 — Calibration draws

The question 4.2–4.3 need answered: how many labelled stimulation-ON trials does a decoder
need before it works again? This notebook builds the data for that experiment. For each
stimulated test session and each k in {2, 5, 10, 20}, it samples 10 sets of k ON trials —
the *calibration draws*. The leftover ON trials of the session are that draw's *query set*,
where any decoder refit on the draw gets scored (per `docs/project-spec.md`).

Two properties make the downstream comparison fair:

- **Nested draws.** Draw `i` at every k is a prefix of one shuffled trial order, so draw
  `i` at k=2 is contained in draw `i` at k=5, 10, 20. One draw index is one growing
  experiment, not four unrelated ones.
- **Reproducibility.** Sampling is seeded per session, so the file reproduces and adding a
  session does not shift the others.

Normalisation uses the drawn trials only: the query set is the leftover ON trials and would
otherwise overlap the statistics it is tested against.

Writes `task4/results/draws.json`.

## Setup

In [ ]:
import os
import sys
import json
import pandas as pd

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
if TASK4_DIR not in sys.path:
    sys.path.insert(0, TASK4_DIR)

from utils import (
    TASK4_DIR,
    TASK4_RESULTS,
    DRAWS_PATH,
    RAW_VARIANT,
    LAG_COUNT,
    STIMULATED_ON,
    build_draws,
    save_draws,
    sessions_in_split,
    iter_draws,
    drawn_trial_windows,
    query_trial_windows,
)

print('task 4 helpers:', TASK4_DIR)
print('draws file   :', DRAWS_PATH)

task 4 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4
draws file   : /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


## Sampling

Build the draws and print the invariant checks `build_draws` runs (set sizes, query sizes,
nestedness) before saving.

In [ ]:
SESSIONS = sessions_in_split('test', STIMULATED_ON) # get the test sessions with stimulation
DRAWS = build_draws(SESSIONS)
print(json.dumps(DRAWS['checks'], indent=1)[:600])
save_draws(DRAWS)
print('wrote', DRAWS_PATH)

{
 "passed": true,
 "failures": [],
 "notes": []
}
wrote /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


## Summary

One row per (session, k): how many distinct sets the 10 draws collapsed to, and how large
the query set is. Query shrinks as k grows — calibration trials come out of the same 50-trial
universe — and at k=20 only 30 query trials are left, which is what 4.3 scores.

In [ ]:
rows = []
for session, entry in DRAWS['sessions'].items():  # one of SESSIONS
    for k in DRAWS['ks']:  # 2, 5, 10, 20 calibration trials
        draws = entry['subsets'][str(k)]  # the 10 sampled draws at this k
        rows.append(dict(session=session, condition=entry['condition'],
                         am_freq_hz=entry['am_freq_hz'], n_on_trials=len(entry['on_trials']), k=k,
                         # distinct sets, so repeated draws at small k are visible
                         distinct_draws=len({frozenset(draw['draw']) for draw in draws}),
                         query_trials=len(draws[0]['query']), n_draws=len(draws)))  # leftover ON trials
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(TASK4_RESULTS, 'task4_1_draws_summary.csv'), index=False)
print('sessions:', summary.session.nunique(), '| rows:', len(summary))
# query shrinks as k grows; the 50-trial universe keeps the 10 draws distinct at every k
summary.groupby('k')[['query_trials', 'distinct_draws']].agg(['min', 'max'])

sessions: 14 | rows: 56


query_trials     distinct_draws    
            min max            min max
k                                     
2            48  48             10  10
5            45  45             10  10
10           40  40             10  10
20           30  30             10  10

## Check that one draw loads

Take the first draw of the first session and confirm the window loader turns it into
calibration and query arrays of the expected shape, with query + calibration = all ON trials.

In [ ]:
session = next(iter(DRAWS['sessions']))
k, index, draw = next(iter_draws(DRAWS, session))
X_cal, y_cal = drawn_trial_windows(session, draw, RAW_VARIANT, LAG_COUNT)
X_query, y_query = query_trial_windows(session, draw, RAW_VARIANT, LAG_COUNT)
print(f'{session} k={k} draw={index}')
print(f'  calibration windows {X_cal.shape} -> {y_cal.shape}')
print(f'  query windows      {X_query.shape} -> {y_query.shape}')
assert len(draw['draw']) == k
assert len(draw['query']) + k == len(DRAWS['sessions'][session]['on_trials'])

Butters-250520-101021 k=2 draw=0
  calibration windows (71, 2800) -> (71, 2)
  query windows      (1696, 2800) -> (1696, 2)
